In [1]:
!pip install opencv-python ultralytics ipywidgets


Defaulting to user installation because normal site-packages is not writeable
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.1.6-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 11.6 MB/s  0:00:01a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.2 MB/s  0:00:00
Using cached nvidia_ml_py-13.610.43-py3-none-any.whl (53 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 kB 10.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 MB 11.6 MB/s  0:00:05 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 11.6 MB/s  0:00:00 11.8 MB/s eta 0:00:01
Using cached ultralytics_thop-2.1.6-py3-none-any.whl (30 kB)
  Attempting uninstall: numpym╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/8 [polars-runtime-32]
    Found existing installation: numpy 2.5.138;5;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/8 [polars-runtime-32]
    Uninstal

In [2]:
import os
import cv2
from ultralytics import YOLO


VIDEO_PATH = "/home/matlab/Downloads/1v.mp4" 


def process_surveillance_video(video_path):
   
    if not os.path.exists(video_path):
        print(f"Error: The file path '{video_path}' does not exist. Please check your spelling and path slashes.")
        return

    
    model = YOLO("yolov8n.pt") 
    
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open the video file: {video_path}")
        return

    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    

    min_seconds_to_process = 30
    max_frames = fps * min_seconds_to_process
    print(f"\n[Video Stats] FPS: {fps} | Resolution: {width}x{height}")
    print(f"[Processing Config] Running tracking loop for {max_frames} frames ({min_seconds_to_process} seconds minimum).")


    output_path = "tracked_surveillance_output.mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_idx = 0
    unique_tracked_objects = set()


    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            print("\nReached the actual end of the video before the 30-second target.")
            break
        

        results = model.track(frame, persist=True, classes=[0, 2, 7], verbose=False)
        

        annotated_frame = results[0].plot()
        out.write(annotated_frame)
        
        print(f"\n--- Processing & Identifying Frame {frame_idx + 1} ---")
        

        if results[0].boxes and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            clss = results[0].boxes.cls.cpu().numpy().astype(int)
            
            for box, obj_id, cls in zip(boxes, ids, clss):
                name = model.names[cls]
                unique_tracked_objects.add(obj_id)
                

                bbox_coords = [round(x, 1) for x in box]
                print(f" -> Found Target: {name} | Tracking ID: #{obj_id} | Bounding Box: {bbox_coords}")
        else:
            print(" -> No specific target objects identified in this frame.")
            
        frame_idx += 1


    cap.release()
    out.release()
    
    print("\n" + "="*40)
    print("SURVEILLANCE METRICS & RESULTS:")
    print(f"Total Unique Targets Tracked: {len(unique_tracked_objects)}")
    print(f"Processed Video Output Saved As: {output_path}")
    print("="*40)


process_surveillance_video(VIDEO_PATH)



[Video Stats] FPS: 29 | Resolution: 2560x1440
[Processing Config] Running tracking loop for 870 frames (30 seconds minimum).
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.7 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 1.3/1.7 MB 9.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 7.0 MB/s  0:00:00

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



/home/matlab/.local/lib/python3.13/site-packages/torch/cuda/__init__.py:1112: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/matlab/.local/lib/python3.13/site-packages/torch/cuda/__init__.py:1170: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count



--- Processing & Identifying Frame 1 ---
 -> Found Target: car | Tracking ID: #1 | Bounding Box: [np.float32(695.7), np.float32(698.6), np.float32(1351.6), np.float32(1042.5)]
 -> Found Target: car | Tracking ID: #2 | Bounding Box: [np.float32(1387.4), np.float32(775.1), np.float32(1967.9), np.float32(1256.8)]
 -> Found Target: person | Tracking ID: #3 | Bounding Box: [np.float32(8.9), np.float32(647.8), np.float32(317.8), np.float32(1204.0)]
 -> Found Target: person | Tracking ID: #4 | Bounding Box: [np.float32(402.9), np.float32(658.4), np.float32(578.8), np.float32(1047.7)]

--- Processing & Identifying Frame 2 ---
 -> Found Target: car | Tracking ID: #1 | Bounding Box: [np.float32(688.6), np.float32(698.2), np.float32(1346.9), np.float32(1043.1)]
 -> Found Target: car | Tracking ID: #2 | Bounding Box: [np.float32(1375.8), np.float32(778.6), np.float32(1962.9), np.float32(1260.8)]
 -> Found Target: person | Tracking ID: #3 | Bounding Box: [np.float32(0.2), np.float32(642.1), np.flo

In [8]:
!pip install ipywidgets opencv-python ultralytics


Defaulting to user installation because normal site-packages is not writeable


In [10]:
import os
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from ultralytics import YOLO

VIDEO_PATH = "/home/matlab/Downloads/1v.mp4"


def is_vehicle_white(frame, bbox):

    h, w, _ = frame.shape
    x1, y1, x2, y2 = map(int, bbox)
    

    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    

    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return False
        

    hsv_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    

    lower_white = np.array([0, 0, 180])   
    upper_white = np.array([180, 45, 255]) 

    mask = cv2.inRange(hsv_crop, lower_white, upper_white)
    

    white_pixel_ratio = np.sum(mask == 255) / mask.size
    

    return white_pixel_ratio > 0.25

def process_and_play_video(video_path):
    if not os.path.exists(video_path):
        print(f"Error: The file path '{video_path}' does not exist.")
        return

    model = YOLO("yolov8n.pt") 
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error: Could not open the video file.")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    max_frames = fps * 30  
 
    image_widget = widgets.Image(format='jpeg', width=640, height=480)
    display(image_widget)

    frame_idx = 0

    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        

        results = model.track(frame, persist=True, classes=[2, 7], verbose=False)
        

        cv2.putText(frame, f"Frame: {frame_idx + 1}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        if results[0].boxes and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            clss = results[0].boxes.cls.cpu().numpy().astype(int)
            
            for box, obj_id, cls in zip(boxes, ids, clss):
  
                if is_vehicle_white(frame, box):
                    x1, y1, x2, y2 = map(int, box)
                    vehicle_type = model.names[cls]
                    

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3) 
                    

                    label_text = f"WHITE {vehicle_type.upper()} | ID: #{obj_id} | F: {frame_idx + 1}"
                    
                    # Position label slightly above the top edge of the bounding box
                    text_y_position = y1 - 10 if y1 - 10 > 20 else y1 + 20
                    
                    # Add an aesthetic background block behind text for high legibility
                    (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                    cv2.rectangle(frame, (x1, text_y_position - th - 4), (x1 + tw, text_y_position + 4), (0, 0, 255), -1)
                    
                    # Render label text
                    cv2.putText(frame, label_text, (x1, text_y_position), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

        # Encode frame as JPEG to instantly stream it directly inside the Notebook UI
        _, jpeg_buffer = cv2.imencode('.jpg', frame)
        image_widget.value = jpeg_buffer.tobytes()
        
        frame_idx += 1

    cap.release()
    print(f"\nProcessing playback finished. Analyzed {frame_idx} video frames total.")

# Fire execution pipeline
process_and_play_video(VIDEO_PATH)


Image(value=b'', format='jpeg', height='480', width='640')

/home/matlab/.local/lib/python3.13/site-packages/torch/cuda/__init__.py:1112: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()



Processing playback finished. Analyzed 900 video frames total.
